In [24]:
import gradio as gr
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import ollama
import re
import os
import yaml
#from langchain.text_splitter import CharacterTextSplitter

In [25]:
with open('config.yaml','r') as file:
    config = yaml.safe_load(file)

In [26]:


model_embedding = config['models']['embedding']
embeddings_directory = config['path']['emmbeddings']
pdf_bytes = config['path']['pdfs']
create_embeddings = config['parameters']['create_embeddings']
chunk_size = config['parameters']['chunk_size']
chunk_overlap = config['parameters']['chunk_overlap']

In [27]:
#model_embedding='all-minilm:l6'
model_embedding ='bge-m3'
#model_embedding = 'deepseek-r1:8b'
#model_embedding ='nomic-embed-text'
#model_chat = 'mistral'      
#model_chat = 'llama3.2'
#model_chat = 'llama3.1:8b'
model_chat = 'deepseek-r1:1.5b'

In [28]:
embeddings_directory = '/datasets/the_coffee_embeddings/'

In [29]:
pdf_bytes = '/datasets/the_coffee_pdf/'

In [30]:
create_embeddings = True

In [31]:
print("Loading PDF documents...")
loader = PyPDFDirectoryLoader(pdf_bytes)
data = loader.load()
print(f"Loaded {len(data)} documents")

Ignoring wrong pointing object 765 0 (offset 0)


Loading PDF documents...
Loaded 70 documents


In [32]:
# Initialize embeddings

embeddings = OllamaEmbeddings(model=model_embedding)



In [33]:
# STEP 1: Clean the document content before splitting
def clean_document_content(docs):
    """Clean OCR errors and normalize text"""
    cleaned_docs = []
    for doc in docs:
        content = doc.page_content
        
        # Common OCR error corrections for Spanish recipes
        corrections = {
            r's:ceased': 'steamed',
            r'bombas de meccia': 'bombas de mezcla',
            r'meccia': 'mezcla',
            r'blend' : 'mezcla',
            r'sirope': 'jarabe',
#            r'Sora': 'Syrup',
            r'Mecia': 'Mezcla',
            r'Aladir': 'Añadir',
            r'tana': 'taza',
            r'sírvala sin más la manga que el cliente la pida': 'sírvala sin tapa a menos que el cliente la pida',
            r'cocer al vapor': 'vaporizar',
            r'demerera': 'morena',
            r'cucharadita': 'cucharadita'
        }
        
        # Apply corrections
        for error, correction in corrections.items():
            content = re.sub(error, correction, content, flags=re.IGNORECASE)
        
        # Remove extra whitespace and normalize line breaks
        content = re.sub(r'\n+', '\n', content)
        content = re.sub(r' +', ' ', content)
        
        # Create new document with cleaned content
        cleaned_doc = doc.copy()
        cleaned_doc.page_content = content.strip()
        cleaned_docs.append(cleaned_doc)
    
    return cleaned_docs

In [34]:
print("Cleaning document content...")
cleaned_data = clean_document_content(data)


Cleaning document content...


/tmp/ipykernel_23/1789809274.py:34: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  cleaned_doc = doc.copy()


In [39]:
# STEP 2: Improved text splitter for recipe structure
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # Increased for recipe context
    chunk_overlap=150,
    separators=["\n\n## ", "\n## ", "\n\n---", "\n---", "\n\n", "\n• ", "\n- ", "\n", ". "]
)


In [40]:
# Spanish-optimized text splitter
#text_splitter = RecursiveCharacterTextSplitter(
#            chunk_size=500,
#            chunk_overlap=150,
#            separators=["\n\n---", "\n---", "## ", "# ", "\n\n", "\n", ". ", "! ", "? "]
 #       )

#text_splitter = CharacterTextSplitter(
#     separator="\n\n---",
#     chunk_size=300,
#     chunk_overlap=50,
#     length_function=len,
#     is_separator_regex=False,
# )

In [41]:
print("Splitting documents...")
docs = text_splitter.split_documents(cleaned_data)
print(f"Created {len(docs)} chunks")


Splitting documents...
Created 9 chunks


In [38]:
# Print sample chunks to verify quality
print("\n=== SAMPLE CLEANED CHUNKS ===")
for i, doc in enumerate(docs[:3]):
    print(f"Chunk {i+1}:\n{doc.page_content[:300]}...\n{'-'*50}")


=== SAMPLE CLEANED CHUNKS ===
Chunk 1:
Ficha técnica - Bebidas...
--------------------------------------------------
Chunk 2:
Preparados/mix 
 Matcha Chai Massala Chocolate Fresas in natura Fresas congelada Limón Cold brew concentrado Azul Sora mezcla Horchata mix Maracuyá...
--------------------------------------------------
Chunk 3:
PURISTAS 
Global Calientes Pure Black (Double Shot) Pure Black (Single Shot) Americano True White Matcha Latte 
Frias Iced Latte Iced Black Matcha Iced Latte Iced Chocolate...
--------------------------------------------------


In [16]:
# Create vector store
print("Creating vector store...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=embeddings_directory
)


Creating vector store...


In [17]:
# STEP 3: Improved retriever with better search parameters
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Use MMR for diverse results
    search_kwargs={
        "k": 4}
)

In [18]:
# Create vector store
#print("Creating vector store...")
#vectorstore = Chroma.from_documents(
#            documents=docs,
#            embedding=embeddings,
#            persist_directory=embeddings_directory
#        )
            
#retriever = vectorstore.as_retriever(
#            search_type="similarity",
#            search_kwargs={"k": 4}
#        )

In [19]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7c04e564e770>, search_kwargs={'k': 4})

In [20]:

def ollama_llm(question, context, model_chat):
    formatted_prompt = f"""Eres un experto en recetas de café y té. Analiza el contexto y responde en español.

    CONTEXTO DE RECETAS:
    {context}

    PREGUNTA: {question}

    INSTRUCCIONES ESPECÍFICAS:
    1. Responde ÚNICAMENTE en español
    2. Si encuentras la receta exacta, proporciona:
       - Nombre de la receta
       - Ingredientes con cantidades exactas
       - Pasos de preparación
       - Temperaturas y medidas específicas
    3. Si no encuentras la receta exacta, busca recetas similares y di: "No tengo la receta exacta, pero aquí hay recetas similares:"
    4. NO inventes ingredientes o pasos
    5. Si el contexto tiene texto corrupto o errores OCR, intenta interpretarlo basándote en patrones de recetas

    FORMATO DE RESPUESTA:
    - Usa listas claras con • para ingredientes
    - Mantén las medidas originales (ml, g, tsp, etc.)
    - Incluye temperaturas cuando estén especificadas"""

    response = ollama.chat(
        model=model_chat,
        messages=[{"role": "user", "content": formatted_prompt}],
        options={
            "temperature": 0.1,
            "top_k": 10,
            "top_p": 0.9
        }
    )
    
    response_content = response["message"]["content"]
    final_answer = re.sub(r"<think>.*?</think>", "", response_content, flags=re.DOTALL).strip()
    return final_answer


In [21]:
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
# STEP 4: Test with specific recipe queries
def test_recipe_queries():
    test_questions = [
        "dame la receta exacta de matcha latte",
        "dame la receta exacta de chai latte", 
        "ingredientes y cantidades para true white "]
    
    for question in test_questions:
        print(f"\n{'='*60}")
        print(f"QUERY: {question}")
        print(f"{'='*60}")
        
        retrieved_docs = retriever.invoke(question)
        print(f"Retrieved {len(retrieved_docs)} documents")
        
        formatted_content = combine_docs(retrieved_docs)
        
        print("\nRETRIEVED CONTEXT (first 500 chars):")
        print(formatted_content[:800] + "..." if len(formatted_content) > 800 else formatted_content)
        
        print("\nANSWER:")
        answer = ollama_llm(question, formatted_content, model_chat)
        print(answer)
        print(f"{'='*60}")


In [23]:
test_recipe_queries()


QUERY: dame la receta exacta de matcha latte
Retrieved 4 documents

RETRIEVED CONTEXT (first 500 chars):
Seasonal Drinks - Spring
Sora Latte
170 ml steamed milk
36 ml Butterﬂy mix
2 pumps sora blend
Hot cup (240 ml)
2 pumps (10 ml) Sora blend
36 ml Mezcla Azul.
170 ml de leche al vapor.
Coloca la taza caliente en la báscula y tara.
En la taza: Añadir 2 bombeos de mezcla Sora.
En la taza: Añadir 36 ml de la mezcla Butterﬂy.
En la jarra : Vapor 170 ml de leche de 65ºC a 70ºC.
Punto de atención: antes de preparar, compruebe la 
personalización de la temperatura de la leche solicitada 
en la tableta por el cliente.
En la taza: Sirva la leche sobre los demás ingredientes y 
sírvala sin tapa (a menos que el cliente la pida).

Seasonal Drinks - Spring
Sora Latte
170 ml steamed milk
36 ml Butterﬂy mix
2 pumps sora mezcla
Hot cup (240 ml)
2 pumps (10 ml) Sora mezcla
36 ml Mezcla Azul.
170 ml de leche ...

ANSWER:
No tengo la receta exacta, pero aquí hay recetas similares:

---

**Receta del So

In [24]:
def extract_recipe_components(docs):
    """Extract structured recipe information"""
    recipes = []
    
    for doc in docs:
        content = doc.page_content
        
        # Extract recipe name
        recipe_name_match = re.search(r'#\s*(.+?)\n|##\s*(.+?)\n', content)
        recipe_name = recipe_name_match.group(1) or recipe_name_match.group(2) if recipe_name_match else "Unknown"
        
        # Extract ingredients with quantities
        ingredients = re.findall(r'(\d+\s*(?:ml|g|tsp|°C)?\s*[^.\n]*?(?:leche|agua|café|matcha|azúcar|miel|espresso|vapor))', content, re.IGNORECASE)
        
        # Extract temperatures
        temperatures = re.findall(r'(\d+°C\s*a\s*\d+°C|\d+°C)', content)
        
        # Extract equipment/tools
        equipment = re.findall(r'(vaso|jarra|taza|báscula|vaporizador)', content, re.IGNORECASE)
        
        recipes.append({
            'name': recipe_name,
            'ingredients': list(set(ingredients)),  # Remove duplicates
            'temperatures': list(set(temperatures)),
            'equipment': list(set(equipment)),
            'content': content[:500]  # First 500 chars for context
        })
    
    return recipes

# Use the extraction function
print("\n=== EXTRACTED RECIPE COMPONENTS ===")
recipe_components = extract_recipe_components(docs[:5])  # First 5 docs
for i, recipe in enumerate(recipe_components):
    print(f"\nRecipe {i+1}: {recipe['name']}")
    print(f"Ingredients: {recipe['ingredients']}")
    print(f"Temperatures: {recipe['temperatures']}")
    print(f"Equipment: {recipe['equipment']}")


=== EXTRACTED RECIPE COMPONENTS ===

Recipe 1: Unknown
Ingredients: []
Temperatures: []
Equipment: []

Recipe 2: Unknown
Ingredients: []
Temperatures: []
Equipment: []

Recipe 3: Unknown
Ingredients: []
Temperatures: []
Equipment: []

Recipe 4: Unknown
Ingredients: []
Temperatures: []
Equipment: []

Recipe 5: Unknown
Ingredients: []
Temperatures: []
Equipment: []


In [24]:
#question ='dame la receta de un matcha latte'
question ='dame la receta de un chai latte'


In [25]:
retrieved_docs = retriever.invoke(question)
print(f"Retrieved {len(retrieved_docs)} documents")
formatted_content = combine_docs(retrieved_docs)

Retrieved 6 documents


In [26]:
print("\n=== RETRIEVED CONTEXT ===")
print(formatted_content[:1000] + "..." if len(formatted_content) > 1000 else formatted_content)



=== RETRIEVED CONTEXT ===
Seasonal Drinks - Spring
Sora Latte
170 ml steamed milk
36 ml Butterﬂy mix
2 pumps sora blend
Hot cup (240 ml)
2 pumps (10 ml) Sora blend
36 ml Mezcla Azul.
170 ml de leche al vapor.
Coloca la taza caliente en la báscula y tara.
En la taza: Añadir 2 bombeos de mezcla Sora.
En la taza: Añadir 36 ml de la mezcla Butterﬂy.
En la jarra : Vapor 170 ml de leche de 65ºC a 70ºC.
Punto de atención: antes de preparar, compruebe la 
personalización de la temperatura de la leche solicitada 
en la tableta por el cliente.
En la taza: Sirva la leche sobre los demás ingredientes y 
sírvala sin tapa (a menos que el cliente la pida).

Seasonal Drinks - Spring
Syrup Latte
170 ml steamed milk
36 ml Butterﬂy mix
2 pumps Syrup blend
Hot cup (240 ml)
2 pumps (10 ml) Syrup blend
36 ml Mezcla Azul.
170 ml de leche al vapor.
Coloca la taza caliente en la báscula y tara.
En la taza: Añadir 2 bombeos de mezcla Syrup.
En la taza: Añadir 36 ml de la mezcla Butterﬂy.
En la jarra : Vapor 17

In [27]:
print("\n=== ANSWER ===")
answer = ollama_llm(question, formatted_content, model_chat)
print(answer)


=== ANSWER ===
La receta del Chai Latte se encuentra en la sección "Local" del contexto proporcionado. Aquí está la receta:

**Nombre de la receta:** Pure Black and White Caffe Latte (no es exactamente un Chai Latte, pero parece ser una variante similar)

**Ingredientes:**

• 170 ml leche al vapor
• 36 ml mezcla Butterﬂy
• 2 bombas de mezcla Sora (o Syrup blend)
• Hot cup (240 ml)

**Pasos de preparación:**

1. Coloca la taza caliente en la báscula y tara.
2. En la taza, añade 2 bombas de mezcla Sora (o Syrup blend).
3. En la taza, añade 36 ml de la mezcla Butterﬂy.
4. En la jarra: vaporiza 170 ml de leche a una temperatura de 65ºC a 70ºC.
5. Punto de atención: antes de preparar, comprueba la personalización de la temperatura de la leche solicitada en la tableta por el cliente.
6. En la taza: sirve la leche sobre los demás ingredientes y sírvela sin tapa (a menos que el cliente lo pida).

**Nota:** La receta no menciona específicamente los ingredientes típicos de un Chai Latte, como l

In [21]:
def debug_retrieval(question, retriever):
    print(f"\n=== DEBUG: Retrieval for '{question}' ===")
    retrieved_docs = retriever.invoke(question)
    
    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- Document {i+1} (Score: {doc.metadata.get('score', 'N/A')}) ---")
        print(f"Content: {doc.page_content[:300]}...")
        print(f"Metadata: {doc.metadata}")
    
    return retrieved_docs

# Test with different questions
test_questions = [
    "chai latte",
    "matcha iced latte", 
    "receta de café",
    "ingredientes matcha"
]

for q in test_questions:
    debug_retrieval(q, retriever)


=== DEBUG: Retrieval for 'chai latte' ===

--- Document 1 (Score: N/A) ---
Content: Local    
Calientes     Pure  Black  and  White Caffe  Latte Chai  Latte 
Frias  Cold  Brew Chai  Iced  Latte Iced  Mocha...
Metadata: {'producer': 'Skia/PDF m136 Google Docs Renderer', 'creationdate': '', 'page': 15, 'title': 'Ficha técnica | Bebidas | the coffee mx', 'creator': 'PyPDF', 'page_label': '16', 'total_pages': 66, 'source': '/datasets/the_coffee_pdf/Ficha técnica _ Bebidas _ the coffee mx (2).pdf'}

--- Document 2 (Score: N/A) ---
Content: Local    
Calientes     Pure  Black  and  White Caffe  Latte Chai  Latte 
Frias  Cold  Brew Chai  Iced  Latte Iced  Mocha...
Metadata: {'creationdate': '', 'page_label': '16', 'page': 15, 'producer': 'Skia/PDF m136 Google Docs Renderer', 'source': '/datasets/the_coffee_pdf/Ficha técnica _ Bebidas _ the coffee mx (2).pdf', 'creator': 'PyPDF', 'total_pages': 66, 'title': 'Ficha técnica | Bebidas | the coffee mx'}

--- Document 3 (Score: N/A) ---
Content: